# 6) Redispatch and N-1 security — Exercises

**Goals:** build the LODFs and verify them, screen every outage against every line, price short-term emergency ratings, and find which reinforcement N-1 security values most.

The setup cell holds the lecture's network, `PTDF`, `flows(dispatch, ptdf_matrix)`, lesson 4's N-0 dispatch `N0_DISPATCH`, and `secure_dispatch(lodf, limits, post_limits, security)` — the lecture's security-constrained dispatch with a limit per line before and after an outage, which needs **your** LODF matrix. Power in MW, costs in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")
logging.getLogger("linopy").setLevel(logging.ERROR)

# ---- the lecture's network, plants and PTDFs ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM, LIMIT = 0.32, 400.0
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")


def ptdf(lines=LINES, reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * lines.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(lines), len(BUSES)))
    A[range(len(lines)), lines.bus0.map(idx)] = 1
    A[range(len(lines)), lines.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=lines.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1
GI = pd.Index(GENS.index, name="gen")
COST = xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[GI])


def flows(dispatch, ptdf_matrix=PTDF):
    return pd.Series(ptdf_matrix.to_numpy() @ (G @ dispatch[GENS.index].to_numpy() - LOAD.to_numpy()), index=ptdf_matrix.index)


def secure_dispatch(lodf, limits=None, post_limits=None, security=True):
    """The lecture's (security-constrained) dispatch with a limit per line before (limits) and after an outage
    (post_limits); both default to 400 MW. Pass your own LODF matrix."""
    limits = pd.Series(LIMIT, index=LINES.index) if limits is None else limits
    post_limits = limits if post_limits is None else post_limits
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    M, c0 = PTDF.to_numpy() @ G, PTDF.to_numpy() @ LOAD.to_numpy()
    f = {l: (p * xr.DataArray(M[i], coords=[GI])).sum() - float(c0[i]) for i, l in enumerate(LINES.index)}
    for l in LINES.index:
        m.add_constraints(f[l] <= limits[l], name=f"upper {l}")
        m.add_constraints(f[l] >= -limits[l], name=f"lower {l}")
    if security:
        for k in LINES.index:
            for l in LINES.index.drop(k):
                post = f[l] + lodf.loc[l, k] * f[k]
                m.add_constraints(post <= post_limits[l], name=f"upper {l} | {k} out")
                m.add_constraints(post >= -post_limits[l], name=f"lower {l} | {k} out")
    m.add_objective((p * COST).sum())
    m.solve(solver_name="highs", output_flag=False)
    return {"cost": float(m.objective.value), "dispatch": p.solution.to_pandas()}


N0_DISPATCH = pd.Series({"wind Velebit": 485.0, "hydro Split": 700.0, "gas Rijeka": 300.0, "coal Osijek": 400.0,
                         "import Zagreb": 15.050505})            # lesson 4's N-0 dispatch, 81,956.3 EUR/h

### Warm-ups

**1. The LODFs**

Write `lodf()` returning the LODF matrix (rows: monitored line ℓ, columns: outaged line k; LODF_kk = −1). Verify it: for lesson 4's N-0 dispatch and every outage k, the predicted flows f_ℓ + LODF_ℓk · f_k must equal the flows recomputed with `ptdf(LINES.drop(k))`.

In [ ]:
def lodf() -> pd.DataFrame:
    """LODF[l, k]: flow change on line l per MW that line k carried before its outage."""
    ...  # YOUR CODE HERE

In [ ]:
LODF = lodf()
f0 = flows(N0_DISPATCH)
for k in LINES.index:
    without_k = LINES.drop(k)
    recomputed = flows(N0_DISPATCH, ptdf(without_k))
    assert np.allclose(f0.drop(k) + LODF.loc[without_k.index, k] * f0[k], recomputed, atol=1e-6), k
assert np.isclose(secure_dispatch(LODF)["cost"], 107913.8, atol=0.1), "the lecture's SCOPF cost"
LODF.round(3)

**2. The full screening matrix**

Write `screening(dispatch, post_limits)` returning a table: one row per outage, one column per monitored line, entries the post-outage loading in % of `post_limits` (empty for the outaged line itself). Count the violations (> 100 %) for lesson 4's N-0 dispatch.

In [ ]:
def screening(dispatch: pd.Series, post_limits=None) -> pd.DataFrame:
    """Post-outage loading (%) of every line for every single line outage."""
    ...  # YOUR CODE HERE

In [ ]:
table = screening(N0_DISPATCH)
violations = int((table > 100 + 1e-6).sum().sum())
assert violations == 12, "every outage overloads two lines"
print("violations:", violations)
table.round(0)

### Core

**3. Short-term emergency ratings**

Lines may carry more than their continuous rating for the minutes an operator needs to act after an outage. Let the post-outage limit be 100 %, 110 %, 120 %, 130 % and 140 % of 400 MW (the pre-outage limit stays 400 MW). Report the cost of the security-constrained dispatch and its excess over the N-0 cost (81,956.3 €/h). Write `emergency_ratings(factors)`.

In [ ]:
def emergency_ratings(factors=(1.0, 1.1, 1.2, 1.3, 1.4)) -> pd.DataFrame:
    """Cost of N-1 security for each post-outage rating factor."""
    ...  # YOUR CODE HERE

In [ ]:
ratings = emergency_ratings()
assert ratings.cost_of_security.is_monotonic_decreasing
assert np.isclose(ratings.loc[1.0, "cost_of_security"], 25957.5, atol=0.5)
ratings.round(1)

**4. Interpretation**

How fast does the cost of security fall with the emergency rating, and from which rating is N-1 security free here? Why? What does an operator need, besides a higher rating, to rely on it?

*Your answer here.*

### Challenge

**5. Which reinforcement is worth most?**

Upgrade **one** line at a time from 400 to 600 MW (both before and after an outage; all others stay at 400 MW). For each, report the cost saving of the N-0 dispatch and of the N-1 dispatch, against their base costs. Write `reinforcements()` and rank the lines by each saving.

In [ ]:
def reinforcements(upgrade_mw: float = 600.0) -> pd.DataFrame:
    """Cost saving (EUR/h) of upgrading each line, without and with N-1 security."""
    ...  # YOUR CODE HERE

In [ ]:
upgrades = reinforcements()
assert (upgrades[["saving_N-0", "saving_N-1"]] >= -1e-6).all().all()
upgrades.round(1)

**6. Interpretation**

Does N-1 security change which reinforcement is worth most? Explain with the lecture's corridors and LODFs. What does that mean for grid planning studies that check only N-0?

*Your answer here.*